# Colab dataset: Coleta, Pré-processamento e Extração de Features.

## 0. Google Drive e API

In [1]:
import os
import shutil
from google.colab import userdata, drive

drive.mount('/content/drive/')

os.chdir("/content/drive/MyDrive/MD")

Mounted at /content/drive/


In [ ]:
api_key = userdata.get('ROBOFLOW_API_KEY')
hf_token = userdata.get('HF_API_KEY')

## 1. Roboflow

In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=api_key)

project = rf.workspace("phenosight-embrapa-ia").project("original-images-uwqav")
version = project.version(5)

dataset = version.download(model_format="folder", location="./dataset_roboflow")

loading Roboflow workspace...
loading Roboflow project...
Generating version still in progress. Progress: 80.79%
Exporting format folder in progress : 76.0%
Version export complete for folder format



Extracting Dataset Version Zip to ./dataset_roboflow in folder:: 100%|██████████| 1012/1012 [00:09<00:00, 106.21it/s]


In [ ]:
dataset = os.path.join("dataset_roboflow")

## 2. Dividir dataset em Densidade e Tamanho

In [2]:
from tqdm import tqdm

# Caminho base do seu dataset original
base_dir = "dataset_roboflow"

# Novos caminhos
dir_densidade = "dataset_densidade"
dir_tamanho = "dataset_tamanho"

classes_originais = {
    'Denso Grande', 'Denso Medio', 'Denso Pequeno',
    'Distribuido Grande', 'Distribuido Medio', 'Distribuido Pequeno'
}

extensoes_imagem = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

tarefas = []

for root, dirs, files in os.walk(base_dir):
    nome_pasta_atual = os.path.basename(root)

    if nome_pasta_atual in classes_originais:
        partes = nome_pasta_atual.split()
        rotulo_densidade = partes[0]
        rotulo_tamanho = partes[1]

        caminho_relativo = os.path.relpath(root, base_dir)
        divisao = caminho_relativo.split(os.sep)[0]

        destino_densidade = os.path.join(dir_densidade, divisao, rotulo_densidade)
        destino_tamanho = os.path.join(dir_tamanho, divisao, rotulo_tamanho)

        os.makedirs(destino_densidade, exist_ok=True)
        os.makedirs(destino_tamanho, exist_ok=True)

        for arq in files:
            if arq.lower().endswith(extensoes_imagem):
                caminho_origem = os.path.join(root, arq)
                tarefas.append((caminho_origem, os.path.join(destino_densidade, arq), os.path.join(destino_tamanho, arq)))

print(f"Total de imagens encontradas: {len(tarefas)}")

if len(tarefas) == 0:
    print("Aviso: Nenhuma imagem encontrada. Verifique se o caminho base_dir e os nomes das pastas coincidem exatamente.")
else:
    for origem, dest_dens, dest_tam in tqdm(tarefas, desc="Processando imagens"):
        shutil.copyfile(origem, dest_dens)
        shutil.copyfile(origem, dest_tam)

    print("\nConcluído com sucesso! Separado em dois datasets.")

Total de imagens encontradas: 1010


Processando imagens: 100%|██████████| 1010/1010 [08:14<00:00,  2.04it/s]


Concluído com sucesso! Separado em dois datasets.


In [3]:
import glob
from sklearn.model_selection import train_test_split

def realizar_split(input_dir, output_dir, classes, extensoes=('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
    X, y = [], []

    # 1. Coleta e validação de caminhos
    for cls in classes:
        pasta_classe = os.path.join(input_dir, cls)
        if not os.path.exists(pasta_classe):
            print(f"Aviso: Diretório não encontrado: {pasta_classe}")
            continue

        caminhos = [
            f for f in glob.glob(os.path.join(pasta_classe, '*'))
            if f.lower().endswith(extensoes)
        ]

        X.extend(caminhos)
        y.extend([cls] * len(caminhos))

    if len(X) == 0:
        print(f"Nenhuma imagem válida encontrada em: {input_dir}")
        return

    print(f"\n--- Iniciando split para: {input_dir} (Total: {len(X)} imagens) ---")

    # 2. Split estratificado (70% treino, 15% validação, 15% teste)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )

    # 3. Pré-criação de todos os diretórios (evita overhead repetitivo de I/O)
    splits = {
        'train': (X_train, y_train),
        'val': (X_val, y_val),
        'test': (X_test, y_test)
    }

    for split_name in splits.keys():
        for cls in classes:
            os.makedirs(os.path.join(output_dir, split_name, cls), exist_ok=True)

    # 4. Cópia dos arquivos com barra de progresso
    for split_name, (arquivos, rotulos) in splits.items():
        desc = f"{os.path.basename(output_dir)} -> {split_name}"
        for f, lbl in tqdm(zip(arquivos, rotulos), total=len(arquivos), desc=desc):
            nome_arq = os.path.basename(f)
            dest = os.path.join(output_dir, split_name, lbl, nome_arq)
            shutil.copyfile(f, dest)

    print(f"Split de {input_dir} concluído com sucesso!")

# --- Definição dos Caminhos ---
base_path = '/content/drive/MyDrive/MD'

# Executar Split para Densidade
realizar_split(
    input_dir=os.path.join(base_path, 'dataset_densidade', 'train'),
    output_dir=os.path.join(base_path, 'dataset_densidade_split'),
    classes=['Denso', 'Distribuido']
)

# Executar Split para Tamanho
realizar_split(
    input_dir=os.path.join(base_path, 'dataset_tamanho', 'train'),
    output_dir=os.path.join(base_path, 'dataset_tamanho_split'),
    classes=['Grande', 'Medio', 'Pequeno']
)


--- Iniciando split para: /content/drive/MyDrive/MD/dataset_densidade/train (Total: 1010 imagens) ---


dataset_densidade_split -> test: 100%|██████████| 152/152 [00:01<00:00, 77.88it/s]


Split de /content/drive/MyDrive/MD/dataset_densidade/train concluído com sucesso!

--- Iniciando split para: /content/drive/MyDrive/MD/dataset_tamanho/train (Total: 1010 imagens) ---


dataset_tamanho_split -> test: 100%|██████████| 152/152 [00:01<00:00, 82.82it/s]

Split de /content/drive/MyDrive/MD/dataset_tamanho/train concluído com sucesso!


## 3. Aumento de Dados

In [ ]:
import cv2
import albumentations as A
import numpy as np

# Definir o pipeline do Albumentations (Ajuste os parâmetros se necessário)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Rotate(limit=15, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5)
])

def balancear_com_albumentations(diretorio_classe, target_count):
    imagens_originais = glob.glob(os.path.join(diretorio_classe, '*.*'))
    num_atual = len(imagens_originais)

    if num_atual >= target_count or num_atual == 0:
        print(f"Ignorando {diretorio_classe} (Atual: {num_atual} | Alvo: {target_count})")
        return

    faltam = target_count - num_atual
    print(f"[{diretorio_classe}] Gerando {faltam} novas imagens...")

    geradas = 0
    while geradas < faltam:
        for img_path in imagens_originais:
            if geradas >= faltam:
                break

            # Ler imagem com OpenCV (converte BGR padrão para RGB para o Albumentations)
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Aplicar transformação
            aumentada = transform(image=img)['image']

            # Voltar para BGR para salvar
            aumentada_bgr = cv2.cvtColor(aumentada, cv2.COLOR_RGB2BGR)

            # Salvar nova imagem
            base_name = os.path.basename(img_path).split('.')[0]
            new_nome = f"{base_name}_aug_{geradas}.jpg"
            cv2.imwrite(os.path.join(diretorio_classe, new_nome), aumentada_bgr)

            geradas += 1

    print(f"[{diretorio_classe}] Balanceamento concluído!")

# --- APLICANDO BALANCEAMENTO ---

# Densidade: O target é a quantidade de imagens de treino da classe 'Distribuido' (~778 * 0.7 = 544)
dir_treino_densidade = f'{base_path}/dataset_densidade_split/train'
balancear_com_albumentations(f'{dir_treino_densidade}/Denso', target_count=544)

# Tamanho: O target é a quantidade de imagens de treino da classe 'Medio' (~859 * 0.7 = 601)
dir_treino_tamanho = f'{base_path}/dataset_tamanho_split/train'
balancear_com_albumentations(f'{dir_treino_tamanho}/Grande', target_count=601)
balancear_com_albumentations(f'{dir_treino_tamanho}/Pequeno', target_count=601)

/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


[/content/drive/MyDrive/MD/dataset_densidade_split/train/Denso] Gerando 382 novas imagens...
[/content/drive/MyDrive/MD/dataset_densidade_split/train/Denso] Balanceamento concluído!
[/content/drive/MyDrive/MD/dataset_tamanho_split/train/Grande] Gerando 583 novas imagens...
[/content/drive/MyDrive/MD/dataset_tamanho_split/train/Grande] Balanceamento concluído!
[/content/drive/MyDrive/MD/dataset_tamanho_split/train/Pequeno] Gerando 513 novas imagens...
[/content/drive/MyDrive/MD/dataset_tamanho_split/train/Pequeno] Balanceamento concluído!


## 4. Extração de Features

In [4]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 11.6 MB/s eta 0:00:00


In [5]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

MODEL_ID = "260525_2231_yolo26s-obb_96_4.pt"
print(f"Carregando modelo: {MODEL_ID}...")

model_yolo = YOLO(os.path.join(MODEL_ID))
pytorch_model = model_yolo.model.to(device)
pytorch_model.eval() # Modo de inferência (desliga dropout, etc.)

# 3. Hook: Dicionário temporário para guardar o vetor gerado na camada intermediária
features_dict = {}

def get_features_hook(module, input, output):
    # Aplica o Global Average Pooling para transformar o mapa 2D em um vetor 1D
    pooled_output = torch.nn.functional.adaptive_avg_pool2d(output, (1, 1)).flatten(1)
    features_dict["embedding"] = pooled_output

# Busca automaticamente a camada SPPF (melhor camada do backbone para features globais)
target_layer = None
for i, layer in enumerate(pytorch_model.model):
    if "SPPF" in layer.__class__.__name__:
        target_layer = layer
        print(f"--> Hook registrado com sucesso na camada {i} ({layer.__class__.__name__})")
        break

# Se não achar por nome, usa a camada 9 como padrão de segurança
if target_layer is None:
    target_layer = pytorch_model.model[9]
    print("--> Hook registrado na camada padrão (9).")

# Conecta o hook na camada escolhida
hook_handle = target_layer.register_forward_hook(get_features_hook)

# 4. Funções de Extração
def extract_image_embeddings_from_dataloader(model, dataloader, device):
    embeddings = []
    labels = []

    with torch.no_grad(): # Desativa cálculo de gradientes (economiza RAM/VRAM)
        for images, batch_labels in dataloader:
            images = images.to(device)

            # Passa a imagem pelo modelo. O hook vai "roubar" a feature automaticamente.
            _ = model(images)

            # Pega o vetor capturado e move para a CPU
            emb = features_dict["embedding"].cpu().numpy()

            embeddings.append(emb)
            labels.extend(batch_labels.numpy())

    # Empilha todos os lotes num único array
    X_embeddings = np.vstack(embeddings)
    y_labels = np.array(labels)

    return X_embeddings, y_labels

def extract_features(model, dataloader, device, dataset, path, class_names):
    X_features, y_labels = extract_image_embeddings_from_dataloader(model, dataloader, device)
    file_paths = [sample[0] for sample in dataset.samples]

    print(f"\n{'='*60}")
    print(f"Partição: {path.upper()}")
    print(f"Shape dos embeddings: {X_features.shape}")
    print(f"{'='*60}")

    # Cria nomes genéricos para as colunas de features (ex: f_000, f_001...)
    feature_names = [f"f_{i:03d}" for i in range(X_features.shape[1])]

    # Monta o DataFrame
    df = pd.DataFrame(X_features, columns=feature_names)
    df["sample_id"] = df.index
    df["label"] = y_labels
    df["class_name"] = [class_names[int(i)] for i in y_labels]
    df["file_path"] = file_paths
    df["original_split"] = path

    # Organiza as colunas colocando os metadados na frente das features
    metadata_cols = ["sample_id", "label", "class_name", "file_path", "original_split"]
    df = df[metadata_cols + feature_names]

    return df

# 5. Transformações
IMG_SIZE = (1024, 1024)

data_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor() # O YOLO espera valores normalizados entre 0 e 1
])

# 6. Processamento dos Datasets
datasets_names = ["densidade", "tamanho"]
paths = ["train", "val", "test"]

all_datasets_features = {}

for dataset_name in datasets_names:
    all_datasets_features[dataset_name] = {}

    for p in paths:
        dataset_path = f"./dataset_{dataset_name}_split/{p}"

        if not os.path.exists(dataset_path):
            print(f"\n[Aviso] Caminho não encontrado, pulando: {dataset_path}")
            continue

        dataset = ImageFolder(root=dataset_path, transform=data_transforms)

        dataloader = DataLoader(
            dataset,
            batch_size=8,
            shuffle=False,
            num_workers=2
        )

        class_names = dataset.classes
        print(f"\n[{dataset_name.upper()}] - {p.upper()} - Classes: {class_names}")

        # Extrai os dados e salva no DataFrame
        df_features = extract_features(
            pytorch_model,
            dataloader,
            device,
            dataset,
            p,
            class_names
        )

        df_features['split'] = p
        all_datasets_features[dataset_name][p] = df_features

    # Junta as partições (train, val, test) em um único arquivo por dataset
    if all_datasets_features[dataset_name]:
        df_completo = pd.concat(all_datasets_features[dataset_name].values(), ignore_index=True)
        csv_filename = f"yolo_features_{dataset_name}.csv"
        df_completo.to_csv(csv_filename, index=False)
        print(f"\n--> DataFrame unificado salvo com sucesso: {csv_filename}")

# 7. Finalização: Remove o hook para liberar a memória e limpar a arquitetura do modelo
hook_handle.remove()
print("\nProcesso finalizado com sucesso!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Usando dispositivo: cuda
Carregando modelo: 260525_2231_yolo26s-obb_96_4.pt...
--> Hook registrado com sucesso na camada 9 (SPPF)

[DENSIDADE] - TRAIN - Classes: ['Denso', 'Distribuido']

Partição: TRAIN
Shape dos embeddings: (707, 512)

[DENSIDADE] - VAL - Classes: ['Denso', 'Distribuido']

Partição: VAL
Shape dos embeddings: (151, 512)

[DENSIDADE] - TEST - Classes: ['Denso', 'Distribuido']

Partição: TEST
Shape dos embeddings: (152, 512)

--> DataFrame unificado salvo com sucesso: yolo_features_densidade.csv

[TAMANHO] - TRAIN - Classes: ['Grande', 'Medio', 'Pequeno']

Partição: TRAIN
Shape dos embeddings: (707, 512)

[TAMANHO] - VAL - Classes: ['Grande', 'Medio', 'Pequeno']

Partição: VAL
Shape